In [1]:
import zarr
from tqdm import tqdm
import pandas as pd
import numpy as np

In [2]:
eve_data = zarr.group(zarr.DirectoryStore("/mnt/sdomlv2_small/sdomlv2_eve.zarr"))["MEGS-A"]

# aia_wavelengths = list(aia_data["2010"].keys())
eve_ions = list(eve_data.keys())[:-1]

eve_profile = {}
eve_nonnull = {}
for ion in tqdm(eve_ions):
    ion_data = eve_data[ion][:]
    nonull_data = ion_data[ion_data>0.]
    eve_nonnull[ion] = nonull_data
    eve_profile[ion] = nonull_data.shape[0] / ion_data.shape[0]
    # print(f"ion: {ion} has {nonull_data.shape[0]} measurements out of {ion_data.shape[0]} total.")

# eve_profile = pd.Series(eve_profile).sort_values(ascending=False)
# eve_profile

100%|██████████| 39/39 [00:00<00:00, 64.86it/s]


In [8]:
eve_data[eve_ions[0]].attrs.keys()


dict_keys(['ion', 'logT', 'wavelength'])

In [9]:
eve_data.attrs.keys()

dict_keys([])

In [4]:
eve_data["Time"][:]

array(['2010-05-01 00:00:10.484', '2010-05-01 00:01:10.484',
       '2010-05-01 00:02:10.484', ..., '2014-05-26 23:57:09.854',
       '2014-05-26 23:58:09.854', '2014-05-26 23:59:09.855'], dtype='<U23')

In [5]:
df_t_eve = pd.DataFrame({'Time': pd.to_datetime(eve_data['Time'][:]), 'idx_eve': np.arange(0, len(eve_data['Time']))})
df_t_eve['Time'] = pd.to_datetime(df_t_eve['Time']).dt.round("6min")
df_t_obs_eve = df_t_eve.drop_duplicates(subset='Time', keep='first').set_index('Time')
print(df_t_obs_eve.shape)
df_t_obs_eve.head()

(356152, 1)


,idx_eve
Time,
2010-05-01 00:00:00,0
2010-05-01 00:06:00,3
2010-05-01 00:12:00,9
2010-05-01 00:18:00,15
2010-05-01 00:24:00,21


In [6]:
df_t_obs_eve_id = df_t_obs_eve.reset_index()
df_t_obs_eve_id.set_index('idx_eve', inplace=True)
df_t_obs_eve_id.reset_index(inplace=True)
df_t_obs_eve_id.head()

,idx_eve,Time
0,0,2010-05-01 00:00:00
1,3,2010-05-01 00:06:00
2,9,2010-05-01 00:12:00
3,15,2010-05-01 00:18:00
4,21,2010-05-01 00:24:00


In [ ]:
ion1 = eve_ions[0]  
ion2 = eve_ions[1]
ion1_data = eve_data[ion1]
ion2_data = eve_data[ion2]
ion1_data[:]
print(ion1_data[:][0:5])

numpy_df = pd.DataFrame({ion1 : ion1_data[:],
                         ion2 : ion2_data[:]})
numpy_df.reset_index(inplace=True)
numpy_df.rename(columns={'index': 'idx_eve'}, inplace=True)
numpy_df.head()


In [ ]:
ion_df = pd.DataFrame(
    {ion: eve_data[ion][:] for ion in eve_ions}
)
ion_df.reset_index(inplace=True)
ion_df.rename(columns={'index': 'idx_eve'}, inplace=True)
ion_df.head()

In [ ]:
result_df = df_t_obs_eve_id.merge(ion_df, on='idx_eve', how='inner')
result_df.head()
result_df.to_csv("/home/willfaw/data/EVE/evedata_6min.csv", index=False)
result_df.to_parquet("/home/willfaw/data/EVE/evedata_6min.parquet", index=False)

In [ ]:
# ofile = open("/home/willfaw/data/EVE/test_6min.csv", "w")
# for row in df_t_obs_eve.iterrows():
#     time = row[0]
#     for ion in eve_ions[0:3]:
#         ion_value = eve_data[ion][row[1][0]]
#         print(row[0], ion, eve_data[ion][row[1][0]])
